In [12]:
# Packages
library(tidyverse)
library(janitor)
library(readr)
library(lubridate)
library(forecast)
library(tsibble)
library(fable)
#install.packages("ISOweek")
library(ISOweek)

In [13]:
# ============================================================
# FluView Phase 2 (zip) -- cleaning/import script
# Files inside zip (typical):
#   - ILINet.csv
#   - ICL_NREVSS_Clinical_Labs.csv
#   - ICL_NREVSS_Public_Health_Labs.csv
#   - ICL_NREVSS_Combined_prior_to_2015_16.csv
# ============================================================

# ---- 1) Unzip to a folder ----
zip_path <- "FluViewPhase2Data.zip"     # adjust path if needed
out_dir  <- "fluview_raw"

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
unzip(zip_path, exdir = out_dir)

# Helper: build a Monday date from YEAR/WEEK using ISO week rules
# Note: CDC FluView uses MMWR weeks; ISO weeks are a close proxy but not identical.
# If your assignment requires exact CDC MMWR week dates, we can swap this out.
week_start_iso <- function(year, week) {
  ISOweek2date(sprintf("%d-W%02d-1", year, week))  # "-1" = Monday
}

# ---- 2) Generic reader for these FluView CSVs (skip the first line) ----
read_fluview_csv <- function(path) {
  read_csv(
    file = path,
    skip = 1,                      # skip the leading descriptive line
    na = c("X", "", "NA"),
    show_col_types = FALSE,
    progress = FALSE,
    trim_ws = TRUE
  ) |>
    clean_names()
}

# ---- 3) Read and clean ILINet ----
ilinet_path <- file.path(out_dir, "ILINet.csv")

ilinet <- read_fluview_csv(ilinet_path) |>
  mutate(
    region_type = as.character(region_type),
    region      = as.character(region),
    year        = as.integer(year),
    week        = as.integer(week),

    week_start  = week_start_iso(year, week),

    percent_weighted_ili   = as.numeric(percent_weighted_ili),
    percent_unweighted_ili = as.numeric(percent_unweighted_ili),

    across(starts_with("age_"), as.integer),
    ilitotal          = as.integer(ilitotal),
    num_of_providers  = as.integer(num_of_providers),
    total_patients    = as.integer(total_patients)
  ) |>
  arrange(region_type, region, year, week)

# Quick sanity checks
stopifnot(all(ilinet$week >= 1 & ilinet$week <= 53, na.rm = TRUE))

# ---- 4) Read and clean NREVSS lab files ----
nrevss_files <- c(
  "ICL_NREVSS_Clinical_Labs.csv",
  "ICL_NREVSS_Public_Health_Labs.csv",
  "ICL_NREVSS_Combined_prior_to_2015_16.csv"
)

nrevss <- nrevss_files |>
  map(\(fn) {
    p <- file.path(out_dir, fn)
    if (!file.exists(p)) return(NULL)

    df <- read_fluview_csv(p) |>
      mutate(
        source_file = fn,
        region_type = as.character(region_type),
        region      = as.character(region),
        year        = as.integer(year),
        week        = as.integer(week),
        week_start  = week_start_iso(year, week)
      )

    # Coerce numeric columns if present
    num_cols <- intersect(
      names(df),
      c(
        "total_specimens", "total_a", "total_b",
        "percent_positive", "percent_a", "percent_b"
      )
    )
    df <- df |> mutate(across(all_of(num_cols), as.numeric))

    df
  }) |>
  list_rbind() |>
  arrange(source_file, region_type, region, year, week)

# ---- 5) Optional: Keep only National (often used for modeling) ----
ilinet_national <- ilinet |> filter(region_type == "National")
nrevss_national <- nrevss |> filter(region_type == "National")

# ---- 6) Save cleaned outputs ----
dir.create("fluview_clean", showWarnings = FALSE)

write_csv(ilinet, "fluview_clean/ilinet_clean.csv")
write_rds(ilinet, "fluview_clean/ilinet_clean.rds")

write_csv(nrevss, "fluview_clean/nrevss_clean.csv")
write_rds(nrevss, "fluview_clean/nrevss_clean.rds")

**New Changes**:

based on Karl's additional data cleaning

In [17]:
# ============================================================
# CELL: Base tsibble (shared by ETS, ARIMA, ARIMAX)
# ============================================================

ilinet_raw <- readr::read_csv(
  "fluview_clean/ilinet_clean.csv",
  na             = c("NA", "", "X"),
  show_col_types = FALSE
)

df_raw <- ilinet_raw |>
  filter(region_type == "National", year >= 2002, year <= 2024) |>
  arrange(week_start) |>
  select(week_start, percent_weighted_ili,
         age_0_4, age_5_24, age_25_49, age_25_64, age_50_64, age_65) |>
  mutate(week_start = as.Date(week_start))

# Deduplicate
n_dups <- sum(duplicated(df_raw$week_start))
if (n_dups > 0) {
  message(n_dups, " duplicate week_start row(s) — averaging within each date.")
  df_raw <- df_raw |>
    group_by(week_start) |>
    summarise(across(where(is.numeric), \(x) mean(x, na.rm = TRUE)),
              .groups = "drop") |>
    arrange(week_start)
}

# Impute pre-2010 age splits
n_imputed <- sum(is.na(df_raw$age_25_49))
if (n_imputed > 0) {
  message(n_imputed, " rows imputed: age_25_49 = age_50_64 = age_25_64 / 2")
  df_raw <- df_raw |>
    mutate(
      age_25_49 = if_else(is.na(age_25_49), age_25_64 / 2, age_25_49),
      age_50_64 = if_else(is.na(age_50_64), age_25_64 / 2, age_50_64)
    )
}

df <- df_raw |>
  select(week_start, percent_weighted_ili,
         age_0_4, age_5_24, age_25_49, age_50_64, age_65) |>
  mutate(week_start = tsibble::yearweek(week_start)) |>
  as_tsibble(index = week_start) |>
  fill_gaps()

# Linear interpolation for any ILI gaps
if (any(is.na(df$percent_weighted_ili))) {
  x <- as.numeric(df$week_start)
  y <- df$percent_weighted_ili
  df$percent_weighted_ili <- stats::approx(
    x = x[!is.na(y)], y = y[!is.na(y)],
    xout = x, method = "linear", rule = 2
  )$y
}

cat("Rows:", nrow(df), "\n")
cat("Range:", as.character(min(df$week_start)),
    "to", as.character(max(df$week_start)), "\n")
cat("Remaining NAs in age columns:\n")
print(colSums(is.na(select(df, starts_with("age_")))))

# Save for downstream notebooks
write_rds(df, "fluview_clean/df_tsibble.rds")

3 duplicate week_start row(s) — averaging within each date.

384 rows imputed: age_25_49 = age_50_64 = age_25_64 / 2



Rows: 1200 
Range: 2002 W01 to 2024 W52 
Remaining NAs in age columns:
   age_0_4   age_5_24  age_25_49  age_50_64     age_65 week_start 
         3          3          3          3          3          0 


In [18]:
# ============================================================
# CELL: ARIMAX covariates — lagged & differenced (ARIMAX only)
# ============================================================
CHOSEN_LAG    <- 1   # update after inspecting CCF plots
HOLDOUT_WEEKS <- 104

df_lagged <- df |>
  mutate(
    d_age_0_4_lag1   = difference(lag(age_0_4,   CHOSEN_LAG)),
    d_age_5_24_lag1  = difference(lag(age_5_24,  CHOSEN_LAG)),
    d_age_25_49_lag1 = difference(lag(age_25_49, CHOSEN_LAG)),
    d_age_50_64_lag1 = difference(lag(age_50_64, CHOSEN_LAG)),
    d_age_65_lag1    = difference(lag(age_65,    CHOSEN_LAG))
  ) |>
  filter(!is.na(d_age_0_4_lag1)) |>
  fill_gaps() |>
  tidyr::fill(
    percent_weighted_ili,
    d_age_0_4_lag1, d_age_5_24_lag1,
    d_age_25_49_lag1, d_age_50_64_lag1, d_age_65_lag1,
    .direction = "down"
  )

n_total      <- nrow(df_lagged)
n_train_init <- n_total - HOLDOUT_WEEKS

train_full <- df_lagged[1:n_train_init, ]
holdout    <- df_lagged[(n_train_init + 1):n_total, ]

cat("Rows after lagging:", nrow(df_lagged), "\n")
cat("Training rows:     ", nrow(train_full), "\n")
cat("Holdout rows:      ", nrow(holdout),    "\n")
cat("Holdout range:     ", as.character(min(holdout$week_start)),
    "to", as.character(max(holdout$week_start)), "\n")

# Save for ARIMAX notebook
write_rds(df_lagged,  "fluview_clean/df_lagged.rds")
write_rds(train_full, "fluview_clean/train_full.rds")
write_rds(holdout,    "fluview_clean/holdout.rds")

Rows after lagging: 1198 
Training rows:      1094 
Holdout rows:       104 
Holdout range:      2023 W01 to 2024 W52 
